# 07. Hyperparameter Tuning

`train_eda_revised.csv`와 `test_eda_revised.csv`로 Logistic Regression, Random Forest, XGBoost, LightGBM의 기본 성능과 Optuna 튜닝 후 성능을 비교한다. 마지막에는 튜닝된 base model로 stacking 성능까지 확인한다.

In [10]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
N_SPLITS = 5
N_TRIALS = 30

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

DATA_DIR = Path('../data')
TRAIN_PATH = DATA_DIR / 'train_eda_revised.csv'
TEST_PATH = DATA_DIR / 'test_eda_revised.csv'

## 데이터 로드 및 전처리

`06_modeling.ipynb`처럼 타깃 누수 가능성이 있거나 식별자에 가까운 컬럼은 먼저 제거한다. 튜닝은 train 내부 cross validation으로 진행하고, 최종 평가는 test 데이터로만 본다.

In [11]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

# 06_modeling.ipynb와 동일하게 정답 이후에 알 수 있는 정보와 식별자 컬럼 제거
drop_cols = [
    'cust_number', 'name_customer', 'clear_date', 'buisness_year', 'due_in_date',
    'posting_id', 'baseline_create_date', 'target_old', 'year_month',
    'year_quarter', 'business_days_late'
]

train = train.drop(columns=[col for col in drop_cols if col in train.columns])
test = test.drop(columns=[col for col in drop_cols if col in test.columns])

# target 분리 후 나머지 컬럼만 모델 입력으로 사용
train_y = train['target'].astype(int)
test_y = test['target'].astype(int)
train_x = train.drop(columns='target')
test_x = test.drop(columns='target')

# 모델 입력에 직접 누수 컬럼이 남아있는지 확인
leakage_check_cols = {
    'clear_date', 'due_in_date', 'baseline_create_date', 'target_old',
    'business_days_late', 'delay_days', 'target', 'posting_id'
}
print('남아있는 누수 의심 컬럼:', sorted(set(train_x.columns) & leakage_check_cols))

# 수치형/범주형 컬럼을 나누어 pipeline에서 일관되게 전처리
cat_cols = train_x.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = train_x.select_dtypes(exclude=['object', 'category']).columns.tolist()

print(f'train shape: {train_x.shape}')
print(f'test shape : {test_x.shape}')
print(f'categorical columns: {len(cat_cols)}, numeric columns: {len(num_cols)}')

남아있는 누수 의심 컬럼: []
train shape: (32000, 45)
test shape : (8000, 45)
categorical columns: 3, numeric columns: 42


In [12]:
def make_preprocessor(scale_numeric=False):
    # 선형 모델은 스케일링, 트리 계열 모델은 원래 스케일 유지
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))

    numeric_transformer = Pipeline(numeric_steps)
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    return ColumnTransformer([
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ])


def make_pipeline(model, scale_numeric=False):
    return Pipeline([
        ('preprocess', make_preprocessor(scale_numeric=scale_numeric)),
        ('model', model)
    ])


# Optuna objective와 stacking OOF 생성에 같은 CV 설정 사용
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
positive = train_y.sum()
negative = len(train_y) - positive
scale_pos_weight = negative / positive

print(f'scale_pos_weight: {scale_pos_weight:.3f}')

scale_pos_weight: 14.023


## 기본 모델 성능

튜닝 전 모델은 기존 modeling 노트북의 설정과 최대한 비슷하게 둔다.

In [13]:
baseline_models = {
    'Logistic Regression': make_pipeline(
        LogisticRegression(max_iter=5000, random_state=SEED, class_weight='balanced'),
        scale_numeric=True
    ),
    'Random Forest': make_pipeline(
        RandomForestClassifier(n_estimators=300, random_state=SEED, class_weight='balanced', n_jobs=-1)
    ),
    'XGBoost': make_pipeline(
        xgb.XGBClassifier(n_estimators=300, random_state=SEED, eval_metric='logloss', n_jobs=-1)
    ),
    'LightGBM': make_pipeline(
        lgb.LGBMClassifier(n_estimators=300, random_state=SEED, verbose=-1, n_jobs=-1)
    )
}


def evaluate_model(name, model):
    # 최종 성능 비교는 test_eda_revised 기준으로 계산
    model.fit(train_x, train_y)
    pred = model.predict(test_x)
    return {
        'model': name,
        'accuracy': accuracy_score(test_y, pred),
        'macro_f1': f1_score(test_y, pred, average='macro')
    }


baseline_results = [evaluate_model(name, model) for name, model in baseline_models.items()]
baseline_df = pd.DataFrame(baseline_results)
baseline_df

,model,accuracy,macro_f1
0,Logistic Regression,0.594000,0.458305
1,Random Forest,0.958875,0.702251
2,XGBoost,0.959375,0.748656
3,LightGBM,0.961750,0.754648


## Optuna 튜닝

`macro F1`을 objective로 사용한다. trial 수는 실행 시간을 고려해 30회로 두고, 더 오래 돌릴 수 있으면 `N_TRIALS`만 늘리면 된다.

In [14]:
best_params = {}


def tune_model(model_name, objective):
    # TPE sampler로 각 모델의 하이퍼파라미터 공간 탐색
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    best_params[model_name] = study.best_params
    print(f'[{model_name}] best macro F1: {study.best_value:.4f}')
    print(study.best_params)
    return study

In [15]:
def objective_lr(trial):
    # Logistic Regression은 규제 강도와 penalty 중심으로 튜닝
    params = {
        'C': trial.suggest_float('C', 1e-3, 10.0, log=True),
        'penalty': trial.suggest_categorical('penalty', ['l1', 'l2']),
        'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced'])
    }
    model = LogisticRegression(
        **params, solver='saga', max_iter=5000, random_state=SEED, n_jobs=-1
    )
    pipe = make_pipeline(model, scale_numeric=True)
    scores = cross_val_score(pipe, train_x, train_y, cv=cv, scoring='f1_macro', n_jobs=-1)
    return scores.mean()


study_lr = tune_model('Logistic Regression', objective_lr)

  0%|          | 0/30 [00:00<?, ?it/s]

[W 2026-05-19 19:29:48,750] Trial 5 failed with parameters: {'C': 1.382623217936987, 'penalty': 'l2', 'class_weight': None} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\rbdnj\anaconda3\envs\invoice\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_21456\3678241045.py", line 12, in objective_lr
    scores = cross_val_score(pipe, train_x, train_y, cv=cv, scoring='f1_macro', n_jobs=-1)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\rbdnj\anaconda3\envs\invoice\Lib\site-packages\sklearn\utils\_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\rbdnj\anaconda3\envs\invoice\Lib\site-packages\sklearn\model_selection\_validation.py", line 651, in cross

KeyboardInterrupt: 

In [ ]:
def objective_rf(trial):
    # Random Forest는 트리 깊이, leaf 조건, feature sampling 중심으로 튜닝
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 700, step=100),
        'max_depth': trial.suggest_categorical('max_depth', [None, 6, 8, 10, 12, 16, 20]),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced', 'balanced_subsample'])
    }
    model = RandomForestClassifier(**params, random_state=SEED, n_jobs=-1)
    pipe = make_pipeline(model)
    scores = cross_val_score(pipe, train_x, train_y, cv=cv, scoring='f1_macro', n_jobs=-1)
    return scores.mean()


study_rf = tune_model('Random Forest', objective_rf)

In [ ]:
def objective_xgb(trial):
    # XGBoost는 learning rate, tree 복잡도, sampling, regularization 함께 탐색
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 800, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True)
    }
    model = xgb.XGBClassifier(
        **params, random_state=SEED, eval_metric='logloss', tree_method='hist',
        scale_pos_weight=scale_pos_weight, n_jobs=-1
    )
    pipe = make_pipeline(model)
    scores = cross_val_score(pipe, train_x, train_y, cv=cv, scoring='f1_macro', n_jobs=-1)
    return scores.mean()


study_xgb = tune_model('XGBoost', objective_xgb)

In [ ]:
def objective_lgbm(trial):
    # LightGBM은 leaf 수와 regularization에 민감하므로 해당 범위 포함
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 800, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'max_depth': trial.suggest_int('max_depth', -1, 16),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 80),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced'])
    }
    model = lgb.LGBMClassifier(**params, random_state=SEED, verbose=-1, n_jobs=-1)
    pipe = make_pipeline(model)
    scores = cross_val_score(pipe, train_x, train_y, cv=cv, scoring='f1_macro', n_jobs=-1)
    return scores.mean()


study_lgbm = tune_model('LightGBM', objective_lgbm)

## 튜닝 전후 성능 비교

In [ ]:
tuned_models = {
    'Logistic Regression': make_pipeline(
        LogisticRegression(
            **best_params['Logistic Regression'], solver='saga', max_iter=5000,
            random_state=SEED, n_jobs=-1
        ),
        scale_numeric=True
    ),
    'Random Forest': make_pipeline(
        RandomForestClassifier(**best_params['Random Forest'], random_state=SEED, n_jobs=-1)
    ),
    'XGBoost': make_pipeline(
        xgb.XGBClassifier(
            **best_params['XGBoost'], random_state=SEED, eval_metric='logloss',
            tree_method='hist', scale_pos_weight=scale_pos_weight, n_jobs=-1
        )
    ),
    'LightGBM': make_pipeline(
        lgb.LGBMClassifier(**best_params['LightGBM'], random_state=SEED, verbose=-1, n_jobs=-1)
    )
}

tuned_results = [evaluate_model(name, model) for name, model in tuned_models.items()]
tuned_df = pd.DataFrame(tuned_results)

comparison_df = pd.concat([
    baseline_df.assign(setting='Before tuning'),
    tuned_df.assign(setting='After tuning')
], ignore_index=True)

comparison_df[['setting', 'model', 'accuracy', 'macro_f1']].sort_values(['model', 'setting'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=comparison_df, x='model', y='accuracy', hue='setting', ax=axes[0])
axes[0].set_title('Accuracy 鍮꾧탳')
axes[0].set_xlabel('')
axes[0].set_ylabel('Accuracy')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(data=comparison_df, x='model', y='macro_f1', hue='setting', ax=axes[1])
axes[1].set_title('Macro F1 鍮꾧탳')
axes[1].set_xlabel('')
axes[1].set_ylabel('Macro F1')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## Stacking 모델

각 base model의 out-of-fold 예측 확률을 meta feature로 만들고, meta model은 Logistic Regression을 사용한다.

In [ ]:
def stacking_predict(base_model_dict):
    # test 예측은 fold별 예측 확률 평균으로 사용
    oof_preds = np.zeros((len(train_x), len(base_model_dict)))
    test_preds = np.zeros((len(test_x), len(base_model_dict)))

    for model_idx, (model_name, model) in enumerate(base_model_dict.items()):
        fold_test_preds = np.zeros((len(test_x), N_SPLITS))
        print(f'[{model_name}] OOF ?앹꽦')

        for fold, (tr_idx, val_idx) in enumerate(cv.split(train_x, train_y), start=1):
            fold_model = clone(model)
            fold_model.fit(train_x.iloc[tr_idx], train_y.iloc[tr_idx])

            oof_preds[val_idx, model_idx] = fold_model.predict_proba(train_x.iloc[val_idx])[:, 1]
            fold_test_preds[:, fold - 1] = fold_model.predict_proba(test_x)[:, 1]

        test_preds[:, model_idx] = fold_test_preds.mean(axis=1)

    meta_model = LogisticRegression(
        penalty='l1', solver='saga', C=1.0, max_iter=3000, random_state=SEED
    )
    meta_model.fit(oof_preds, train_y)
    final_pred = meta_model.predict(test_preds)
    final_prob = meta_model.predict_proba(test_preds)[:, 1]

    return final_pred, final_prob, meta_model


baseline_stack_pred, baseline_stack_prob, baseline_meta = stacking_predict(baseline_models)
tuned_stack_pred, tuned_stack_prob, tuned_meta = stacking_predict(tuned_models)

In [ ]:
stacking_df = pd.DataFrame([
    {
        'model': 'Stacking - Before tuning',
        'accuracy': accuracy_score(test_y, baseline_stack_pred),
        'macro_f1': f1_score(test_y, baseline_stack_pred, average='macro')
    },
    {
        'model': 'Stacking - After tuning',
        'accuracy': accuracy_score(test_y, tuned_stack_pred),
        'macro_f1': f1_score(test_y, tuned_stack_pred, average='macro')
    }
])

final_compare_df = pd.concat([
    comparison_df[['model', 'accuracy', 'macro_f1']].assign(group=comparison_df['setting']),
    stacking_df.assign(group='Stacking')
], ignore_index=True)

stacking_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.barplot(data=stacking_df, x='model', y='accuracy', ax=axes[0])
axes[0].set_title('Stacking Accuracy')
axes[0].set_xlabel('')
axes[0].set_ylabel('Accuracy')
axes[0].tick_params(axis='x', rotation=15)

sns.barplot(data=stacking_df, x='model', y='macro_f1', ax=axes[1])
axes[1].set_title('Stacking Macro F1')
axes[1].set_xlabel('')
axes[1].set_ylabel('Macro F1')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
print('[Tuned Stacking] classification report')
print(classification_report(test_y, tuned_stack_pred))

cm = confusion_matrix(test_y, tuned_stack_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Tuned Stacking Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
meta_feature_names = ['LR', 'RF', 'XGB', 'LGBM']
meta_importance = pd.Series(tuned_meta.coef_[0], index=meta_feature_names).sort_values(key=abs, ascending=False)

plt.figure(figsize=(7, 4))
colors = ['#4C78A8' if value >= 0 else '#F58518' for value in meta_importance]
sns.barplot(x=meta_importance.values, y=meta_importance.index, palette=colors)
plt.axvline(0, color='black', linewidth=1)
plt.title('Tuned Stacking Meta Model 계수')
plt.xlabel('Coefficient')
plt.ylabel('Base model')
plt.tight_layout()
plt.show()

meta_importance